In [1]:
import torch
import torch.nn as nn

class LoRALayer(nn.Module):
    def __init__(self, in_dim, out_dim, rank, alpha):
        super().__init__()
        std_dev=1/torch.sqrt(torch.tensor(rank).float())
        self.A=nn.Parameter(torch.randn(in_dim, rank)*std_dev)
        self.B=nn.Parameter(torch.zeros(rank, out_dim))
        self.alpha=alpha

    def forward(self, x):
        x = x @ self.A @ self.B # Compute xA @ B
        return x

class LinearWithLoRA(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear=linear
        self.lora=LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        return self.linear(x)+self.lora(x)

# Hyperparameters
random_seed=123

torch.manual_seed(random_seed)

# Exercise 3: Creating a Small Neural Network and Applying LoRA
# Define a single-layer neural network using nn.Linear.
layer=nn.Linear(10, 5)
x=torch.randn(1, 10)

print(x)
print(layer)
print('Original output:', layer(x))

## Applying LoRA to Linear Layer
rank = 4
alpha = 8
layer_lora_1=LinearWithLoRA(layer, rank, alpha)
print(layer_lora_1(x))


import torch.nn.functional as F

# This LoRA code is equivalent to LinearWithLoRA
class LinearWithLoRAMerged(nn.Module):
    def __init__(self, linear, rank, alpha):
        super().__init__()
        self.linear=linear
        self.lora=LoRALayer(
            linear.in_features, linear.out_features, rank, alpha
        )

    def forward(self, x):
        lora_weights = self.lora.A @ self.lora.B # combine LoRA metrices
        # then combine LoRA original weights
        combined_weight=self.linear.weight + (self.lora.alpha / self.lora.A.shape[1]) * lora_weights.T # Scaling with alpha/rank
        return F.linear(x, combined_weight, self.linear.bias)

# Exercise 4: Merging LoRA Matrices and Testing Equivalence
layer_lora_2=LinearWithLoRAMerged(layer, rank, alpha)
print(layer_lora_2(x))

class MultilayerPerceptron(nn.Module):
    def __init__(self, num_features, num_hidden_1, num_hidden_2, num_classes):
        super().__init__()
        self.layers=nn.Sequential(
          nn.Linear(num_features, num_hidden_1),
          nn.ReLU(),
          nn.Linear(num_hidden_1, num_hidden_2),
          nn.ReLU(),
          nn.Linear(num_hidden_2, num_classes)
        )

    def forward(self, x):
        x=self.layers(x)
        return x

# Architecture
num_features=784 # For MNIST 28x28 images
num_hidden_1=128
num_hidden_2=64
num_classes=10 # For MNIST digits 0-9

# Settings
DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
learning_rate=0.001
num_epochs=10

model=MultilayerPerceptron(
    num_features=num_features,
    num_hidden_1=num_hidden_1,
    num_hidden_2=num_hidden_2,
    num_classes=num_classes
)

model.to(DEVICE)
optimizer_pretrained=torch.optim.Adam(model.parameters(), lr=learning_rate)
print(DEVICE)
print(model)
print(optimizer_pretrained)

"""## Loading dataset"""

from torchvision import datasets
from torchvision import transforms
from torch.utils.data import DataLoader

BATCH_SIZE=64

# Note: transforms.ToTensor() scales input images to 0-1 range
train_dataset=datasets.MNIST(root='data', train=True, transform=transforms.ToTensor(), download=True)

test_dataset=datasets.MNIST(root='data', train=False, transform=transforms.ToTensor(), download=True)

train_loader=DataLoader(dataset=train_dataset, batch_size=BATCH_SIZE, shuffle=True)

test_loader=DataLoader(dataset=test_dataset, batch_size=BATCH_SIZE, shuffle=False)

for images, labels in train_loader:
    print('Image batch dimensions:', images.shape)
    print('Image label dimensions:', labels.shape)
    break

"""## Define evaluation"""

def compute_accuracy(model, data_loader, device):
    model.eval()
    correct_pred, num_examples=0,0
    with torch.no_grad():
        for features, targets in data_loader:
            features=features.view(-1, 28*28).to(device)
            targets=targets.to(device)
            logits=model(features)
            _, predicted_labels=torch.max(logits,1)
            num_examples+=targets.size(0)
            correct_pred+=(predicted_labels == targets).sum().item()
        return correct_pred / num_examples * 100

"""## Training"""

import time

def train(num_epochs, model, optimizer, train_loader, device):
    start_time=time.time()
    for epoch in range(num_epochs):
        model.train()
        for batch_idx, (features, targets) in enumerate(train_loader):
            features=features.view(-1, 28*28).to(device)
            targets=targets.to(device)

            # forward and back propagation
            logits=model(features)
            loss=F.cross_entropy(logits, targets)
            optimizer.zero_grad()

            loss.backward()

            # update model parameters
            optimizer.step()

            # logging
            if not batch_idx %400:
                print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))

        with torch.set_grad_enabled(False):
            print('Epoch: %03d/%03d training accuracy: %.2f%%' % (epoch+1, num_epochs, compute_accuracy(model, train_loader, device)))

        print('Time elapsed: %.2f min' % ((time.time() - start_time)/60))
    print('Total Training Time: %.2f min' % ((time.time() - start_time)/60))


train(num_epochs, model, optimizer_pretrained, train_loader, DEVICE)
print(f'Test accuracy: {compute_accuracy(model, test_loader, DEVICE):.2f}%')

"""# Replacing Linear with LoRA Layers

Using $LinearWithLoRA$, we can then add the LoRA layers by replacing the original $Linear$ layers in the multilayer perception model:
"""

import copy

model_lora=copy.deepcopy(model)

# Exercise 5: Implementing a Multilayer Perceptron (MLP) and Replacing Layers with LoRA
model_lora.layers[0]=LinearWithLoRAMerged(model_lora.layers[0], rank=rank, alpha=alpha)
model_lora.layers[2]=LinearWithLoRAMerged(model_lora.layers[2], rank=rank, alpha=alpha)
model_lora.layers[4]=LinearWithLoRAMerged(model_lora.layers[4], rank=rank, alpha=alpha)
model_lora.to(DEVICE)
optimizer_lora=torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
print(model_lora)

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

"""## Freezing the Original Linear Layers

Then, we can freeze the original $Lienar$ layers and only make the $LoRALayer$ layers trainable, as follows:
"""


def freeze_linear_layers(model):
    for child in model.children():
        if isinstance(child, nn.Linear):
            for param in child.parameters():
                param.requires_grad=False
        else:
            # recursively freeze linear layers in children modules
            freeze_linear_layers(child)

freeze_linear_layers(model_lora)
for name, param in model_lora.named_parameters():
    print(f'{name}:{param.requires_grad}')

"""Based on the True and False values above, we can visually confirm that only the LoRA layers are trainble now(**True means trainable, False means frozen**). In practice, we would then train the network with this LoRA configuration on a new dataset or task. Before we do this, let understand DoRA first."""

optimizer_lora=torch.optim.Adam(model_lora.parameters(), lr=learning_rate)
train(num_epochs, model_lora, optimizer_lora, train_loader, DEVICE)
print(f'Test accuracy LoRA finetune: {compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

print(f'Test accuracy orig model:{compute_accuracy(model, test_loader, DEVICE):.2f}%')
print(f'Test accuracy LoRA model:{compute_accuracy(model_lora, test_loader, DEVICE):.2f}%')

tensor([[-0.2089,  1.0131,  0.0520, -0.8516, -0.2075,  0.5540, -1.8872, -0.7532,
         -1.6264,  0.5861]])
Linear(in_features=10, out_features=5, bias=True)
Original output: tensor([[ 1.2565,  0.8322, -0.5691,  0.9849, -0.7356]],
       grad_fn=<AddmmBackward0>)
tensor([[ 1.2565,  0.8322, -0.5691,  0.9849, -0.7356]], grad_fn=<AddBackward0>)
tensor([[ 1.2565,  0.8322, -0.5691,  0.9849, -0.7356]],
       grad_fn=<AddmmBackward0>)
cpu
MultilayerPerceptron(
  (layers): Sequential(
    (0): Linear(in_features=784, out_features=128, bias=True)
    (1): ReLU()
    (2): Linear(in_features=128, out_features=64, bias=True)
    (3): ReLU()
    (4): Linear(in_features=64, out_features=10, bias=True)
  )
)
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


100%|██████████| 9.91M/9.91M [00:00<00:00, 44.6MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 1.08MB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 9.54MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 3.61MB/s]


Image batch dimensions: torch.Size([64, 1, 28, 28])
Image label dimensions: torch.Size([64])
Epoch: 001/010|Batch 000/938| Loss: 2.2850


/tmp/ipykernel_3594/3530368631.py:169: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:836.)
  print('Epoch: %03d/%03d|Batch %03d/%03d| Loss: %.4f' % (epoch+1, num_epochs, batch_idx, len(train_loader), loss))


Epoch: 001/010|Batch 400/938| Loss: 0.4060
Epoch: 001/010|Batch 800/938| Loss: 0.0642
Epoch: 001/010 training accuracy: 95.08%
Time elapsed: 0.30 min
Epoch: 002/010|Batch 000/938| Loss: 0.0828
Epoch: 002/010|Batch 400/938| Loss: 0.0561
Epoch: 002/010|Batch 800/938| Loss: 0.1065
Epoch: 002/010 training accuracy: 96.96%
Time elapsed: 0.59 min
Epoch: 003/010|Batch 000/938| Loss: 0.0559
Epoch: 003/010|Batch 400/938| Loss: 0.0570
Epoch: 003/010|Batch 800/938| Loss: 0.1057
Epoch: 003/010 training accuracy: 97.93%
Time elapsed: 0.89 min
Epoch: 004/010|Batch 000/938| Loss: 0.0163
Epoch: 004/010|Batch 400/938| Loss: 0.1230
Epoch: 004/010|Batch 800/938| Loss: 0.0607
Epoch: 004/010 training accuracy: 98.12%
Time elapsed: 1.18 min
Epoch: 005/010|Batch 000/938| Loss: 0.0209
Epoch: 005/010|Batch 400/938| Loss: 0.0387
Epoch: 005/010|Batch 800/938| Loss: 0.0719
Epoch: 005/010 training accuracy: 98.65%
Time elapsed: 1.49 min
Epoch: 006/010|Batch 000/938| Loss: 0.0868
Epoch: 006/010|Batch 400/938| Loss: